In [30]:
# %%
import numpy as np
import json
import chempy as chem
from chempy.util import periodic as per
import pandas as pd
from tqdm import tqdm

from chempy import Substance
import periodictable
import tabulate

# Material and Elements

In [31]:
periodic_table = {per.atomic_number(sym): sym for sym in per.symbols}
material_names = ['Carbon', 'Water', 'Quartz', 'Feldspar', 'Mica']
material_formulas = ['C', 'H2O', 'SiO2', 'NaAlO2(SiO2)3', 'K2Al2O5(Si2O5)3Al4(OH)4']
substances = [Substance.from_formula(formula) for formula in material_formulas]
compositions = [substance.composition for substance in substances]
# replace the keys of the compositions with the atomic symbols
compositions = [
	{periodic_table[atomic_number]: value for atomic_number, value in composition.items()}
	for composition in compositions
]
mass_fractions = [chem.mass_fractions(composition) for composition in compositions]
material_frame = pd.DataFrame(mass_fractions, index=material_names)
# make nans zero
material_frame = material_frame.fillna(0)
material_frame_percent = material_frame * 100
material_frame_percent.index.name = 'Material'
material_frame_percent.columns = [column + ' %' for column in material_frame_percent.columns]

In [32]:
print(tabulate.tabulate(material_frame_percent, 
    headers='keys', tablefmt='pipe', showindex=True, floatfmt=".1f"))

| Material   |   C % |   H % |   O % |   Si % |   Na % |   Al % |   K % |
|:-----------|------:|------:|------:|-------:|-------:|-------:|------:|
| Carbon     | 100.0 |   0.0 |   0.0 |    0.0 |    0.0 |    0.0 |   0.0 |
| Water      |   0.0 |  11.2 |  88.8 |    0.0 |    0.0 |    0.0 |   0.0 |
| Quartz     |   0.0 |   0.0 |  53.3 |   46.7 |    0.0 |    0.0 |   0.0 |
| Feldspar   |   0.0 |   0.0 |  48.8 |   32.1 |    8.8 |   10.3 |   0.0 |
| Mica       |   0.0 |   0.5 |  48.2 |   21.2 |    0.0 |   20.3 |   9.8 |


In [33]:
latex_table = tabulate.tabulate(material_frame_percent, headers='keys', tablefmt='latex', showindex=True, floatfmt=".1f")
print(latex_table)

\begin{tabular}{lrrrrrrr}
\hline
 Material   &   C \% &   H \% &   O \% &   Si \% &   Na \% &   Al \% &   K \% \\
\hline
 Carbon     & 100.0 &   0.0 &   0.0 &    0.0 &    0.0 &    0.0 &   0.0 \\
 Water      &   0.0 &  11.2 &  88.8 &    0.0 &    0.0 &    0.0 &   0.0 \\
 Quartz     &   0.0 &   0.0 &  53.3 &   46.7 &    0.0 &    0.0 &   0.0 \\
 Feldspar   &   0.0 &   0.0 &  48.8 &   32.1 &    8.8 &   10.3 &   0.0 \\
 Mica       &   0.0 &   0.5 &  48.2 &   21.2 &    0.0 &   20.3 &   9.8 \\
\hline
\end{tabular}


In [34]:
elements = material_frame.columns.tolist()
atomic_numbers = [per.atomic_number(element) for element in elements]
mcnp_identifiers = [1000*(atomic_number) for atomic_number in atomic_numbers]

translate_dict = {
    1000: 1001,  # Hydrogen
    8000: 8016,  # Oxygen
    14000: 14000,  # Silicon
    6000: 6000,  # Carbon 
    11000: 11023,
    12000: 12000,
    13000: 13027,
}
mcnp_identifiers = [translate_dict.get(x, x) for x in mcnp_identifiers]
# get the density of each element in g/cm^3
densities = [periodictable.elements.symbol(elem).density for elem in elements]
elemental_frame = pd.DataFrame({
    'MCNP Identifier': mcnp_identifiers,
    # 'Density (g/cm^3)': densities
}, index=elements)
_elemental_frame = elemental_frame.copy()
_elemental_frame.index.name = 'Element'
_elemental_frame.columns.name = 'Element'

In [35]:

print(tabulate.tabulate(_elemental_frame.T, 
      headers='keys', tablefmt='pipe', showindex=True))

| Element         |    C |    H |    O |    Si |    Na |    Al |     K |
|:----------------|-----:|-----:|-----:|------:|------:|------:|------:|
| MCNP Identifier | 6000 | 1001 | 8016 | 14000 | 11023 | 13027 | 19000 |


In [36]:
latex_table = tabulate.tabulate(_elemental_frame.T , headers='keys', tablefmt='latex', showindex=True)
print(latex_table)

\begin{tabular}{lrrrrrrr}
\hline
 Element         &    C &    H &    O &    Si &    Na &    Al &     K \\
\hline
 MCNP Identifier & 6000 & 1001 & 8016 & 14000 & 11023 & 13027 & 19000 \\
\hline
\end{tabular}


In [37]:
densities = np.linspace(1.04, 1.59, 3)
densities = densities[1:2]

In [38]:
low_carbon_levels = [i/100 for i in range(0, 16)]        # 0.00 .. 0.15
# high_carbon_levels = [ (5*i)/100 for i in range(4,21-4)]  # 0.20 .. 1.00
# carbon_levels = low_carbon_levels + high_carbon_levels
carbon_levels = low_carbon_levels

In [39]:
hydration_levels = [i/100 for i in np.arange(0, 21, 5).tolist()]

# Soil Material functions

In [40]:
def soil_characteristic_function(
        X, 
        character=None, 
        elem_labels=elements
        ):
    n = X.shape[0]
    out = np.zeros((n, len(elem_labels)))
    if character:
        elem_index = elem_labels.index(character)
        out[:, elem_index] = 1

        out = np.clip(out, 0, None)
        row_sums = out.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        out = out / row_sums
    else:
        pass
    return out
def material_characteristic_function(X, character=None, material_frame=material_frame):
    n = X.shape[0]
    out = np.zeros((n, len(material_frame.columns)))
    if character:
        out[:, :] = material_frame.loc[character].values

        out = np.clip(out, 0, None)
        row_sums = out.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        out = out / row_sums
    else:
        pass
    return out

def material_to_carbon_mechanical_mix(X, character, carbon_fraction=0, material_frame=material_frame):
    n = X.shape[0]
    out = np.zeros((n, len(material_frame.columns)))
    if character:
        out += material_characteristic_function(X, character, material_frame)* (1 - carbon_fraction)
        out += material_characteristic_function(X, 'Carbon', material_frame) * carbon_fraction

        out = np.clip(out, 0, None)
        row_sums = out.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        out = out / row_sums
    else:
        pass
    return out

def material_to_water_mechanical_mix(X, character, water_fraction=0, material_frame=material_frame):
    n = X.shape[0]
    out = np.zeros((n, len(material_frame.columns)))
    if character:
        out += material_characteristic_function(X, character, material_frame) * (1 - water_fraction)
        out += material_characteristic_function(X, 'Water', material_frame) * water_fraction

        out = np.clip(out, 0, None)
        row_sums = out.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        out = out / row_sums
    else:
        pass
    return out

def material_carbon_water_mechanical_mix(X, character=None, carbon_fraction=0, water_fraction=0, material_frame=material_frame):
    n = X.shape[0]
    out = np.zeros((n, len(material_frame.columns)))
    if carbon_fraction < 0 or water_fraction < 0:
        raise ValueError("Carbon and water fractions must be non-negative.")
    if carbon_fraction + water_fraction > 1:
        raise ValueError("Sum of carbon and water fractions must not exceed 1.")
    if character:
        out += material_characteristic_function(X, character, material_frame) * (1 - carbon_fraction - water_fraction)
        out += material_characteristic_function(X, 'Carbon', material_frame) * carbon_fraction
        out += material_characteristic_function(X, 'Water', material_frame) * water_fraction
        
        out = np.clip(out, 0, None)
        row_sums = out.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        out = out / row_sums
    else:
        pass
    return out

def density_char(X, density=1.6):
    n = X.shape[0]
    out = np.zeros(n)
    out[:] = density
    return out

# Simulations

In [41]:
simulations = []

In [42]:
for elem in elements:
    simulation = {}
    simulation['label'] = f"{elem}"
    simulation['elem'] = elem
    simulation['chem_function']  = lambda X, elem=elem: soil_characteristic_function(X, character=elem, elem_labels=elements)
    simulation['chem_function_name'] = f"soil_characteristic_function_{elem}"
    simulation['density_function'] = lambda X, density=np.average(densities): density_char(X, density=density)
    simulation['density_function_name'] = f"density_char_{np.average(densities)}"
    simulation['resolution'] = (1, 1, 1)
    simulations.append(simulation)

In [43]:

simulation = {}
label = f"Water"
simulation['label'] = label
simulation['material'] = 'Water'
simulation['chem_function'] = lambda X, material='Water': material_carbon_water_mechanical_mix(
    X, character=material, carbon_fraction=0, water_fraction=0, material_frame=material_frame)
simulation['chem_function_name'] = f"Water"
simulation['density_function'] = lambda X, density=np.average(densities): density_char(X, density=density)
simulation['density_function_name'] = f"density_char_{str(np.average(densities)).replace('.', 'p')}"

simulation['resolution'] = (1, 1, 1)
simulations.append(simulation)

In [44]:

# simulation = {}
# label = f"Carbon"
# simulation['label'] = label
# simulation['material'] = 'Carbon'
# simulation['chem_function'] = lambda X, material='Carbon': material_carbon_water_mechanical_mix(
#     X, character=material, carbon_fraction=0, water_fraction=0, material_frame=material_frame)
# simulation['chem_function_name'] = f"Carbon"
# simulation['density_function'] = lambda X, density=np.average(densities): density_char(X, density=density)
# simulation['density_function_name'] = f"density_char_{str(np.average(densities)).replace('.', 'p')}"

# simulation['resolution'] = (9, 9, 9)
# simulations.append(simulation)

In [45]:
for material in material_names[2:]:  # Skip Carbon and Water
    for density in densities:
        for carbon_level in carbon_levels:
            for hydration_level in hydration_levels:
                simulation = {}
                label = f"M{material}_D{str(density).replace('.', 'p')}_C{str(carbon_level).replace('.', 'p')}_H{str(hydration_level).replace('.', 'p')}"
                simulation['label'] = label
                simulation['material'] = material
                simulation['carbon_level'] = carbon_level
                simulation['hydration_level'] = hydration_level
                # simulation['chem_function'] = lambda X, material=material, carbon_level=carbon_level: material_to_carbon_mechanical_mix(X, character=material, carbon_fraction=carbon_level, material_frame=material_frame)
                simulation['chem_function'] = lambda X, material=material, carbon_level=carbon_level, hydration_level=hydration_level: material_carbon_water_mechanical_mix(
                    X, character=material, carbon_fraction=carbon_level, water_fraction=hydration_level, material_frame=material_frame)
                simulation['chem_function_name'] = f"material_to_carbon_mechanical_mix_{material}_{str(carbon_level).replace('.', 'p')}"
                simulation['density_function'] = lambda X, density=density: density_char(X, density=density)
                simulation['density_function_name'] = f"density_char_{str(density).replace('.', 'p')}"

                simulation['resolution'] = (1, 1, 1)
                simulations.append(simulation)

# MCNP Code

In [46]:
import numpy as np
from mcnp.DigMat import make_sections, sample_sections
import mcnp

In [47]:
def GenSoil(
        elements,
        element_f, density_f, 
        x_min, x_max, x_n, 
        y_min, y_max, y_n, 
        z_min, z_max, z_n, 
        n_samples=100, origin=(0, 0, 0),
        cell_id_min = 1000,
        surf_id_min = 2000000,
        mat_id_min = 3000000,
        ):

    x_min, x_max, x_n = x_min-origin[0], x_max-origin[0], x_n
    y_min, y_max, y_n = y_min-origin[1], y_max-origin[1], y_n
    z_min, z_max, z_n = z_min-origin[2], z_max-origin[2], z_n

    x_walls = np.linspace(x_min, x_max, x_n + 1)
    y_walls = np.linspace(y_min, y_max, y_n + 1)
    z_walls = np.linspace(z_min, z_max, z_n + 1)
    
    walls = (x_walls, y_walls, z_walls)

    sections = make_sections(x_walls, y_walls, z_walls)

    n = sections.shape[0]

    _ = surf_id_min
    x_surf_ids = np.arange(_, _ + x_n + 1)
    _ = _ + x_n + 1
    y_surf_ids = np.arange(_, _ + y_n + 1)
    _ = _ + y_n + 1
    z_surf_ids = np.arange(_, _ + z_n + 1)
    _ = _ + z_n + 1

    wall_ids = (x_surf_ids, y_surf_ids, z_surf_ids)
    section_wall_ids = make_sections(x_surf_ids, y_surf_ids, z_surf_ids)

    x_surfaces = [mcnp.Surfaces.px(id, x) for id, x in zip(x_surf_ids, x_walls)]
    y_surfaces = [mcnp.Surfaces.py(id, y) for id, y in zip(y_surf_ids, y_walls)]
    z_surfaces = [mcnp.Surfaces.pz(id, z) for id, z in zip(z_surf_ids, z_walls)]

    soil_surfaces = x_surfaces + y_surfaces + z_surfaces
    
    element_mat = sample_sections(element_f, sections, n_samples=n_samples)
    density_mat = sample_sections(density_f, sections, n_samples=n_samples)

    material_ids = np.arange(mat_id_min, mat_id_min + n, dtype=int)
    soil_materials = [mcnp.mcnp.Material(id, elements, elems) for id, elems in zip(material_ids, element_mat)]

    # return section_wall_ids
    surface_strings = [f"{x_i} -{x_ip1} {y_i} -{y_ip1} {z_i} -{z_ip1}" for (x_i, x_ip1, y_i, y_ip1, z_i, z_ip1) in section_wall_ids]

    soil_cells = [mcnp.mcnp.Cell(id, mat_id, dens, surface_string=surf) for id, mat_id, dens, surf in zip(range(cell_id_min, cell_id_min + n), material_ids, density_mat, surface_strings)]

    return (soil_cells, soil_surfaces, soil_materials), (element_mat, density_mat, sections, section_wall_ids), (walls, wall_ids)

In [48]:
def Build2025(offset, distance, height):
    # Shielding
    detector_cell = mcnp.mcnp.Cell(
        cell_id=101,
        material_id=301,
        density=5.08,
        surface_string='-201',
        label='Detector'
    )
    detector_surface = mcnp.mcnp.Surfaces.rcc(surface_id=201, vx=56, vy=-5, vz=-1, h1=0, h2=20, h3=0.0, r=4.5)
    detector_material = mcnp.mcnp.Material(
        material_id=301,
        elements=['35079', '35081', '57139', '58140'],
        portions=[0.2946, 0.3069, 0.3485, 0.0500],
        labels=['Br79', 'Br81', 'La139', 'Ce140'],
        label='Detector'
    )

    det_cells = [detector_cell]
    det_surfaces = [detector_surface]
    det_materials = [detector_material]

    # PE+Pb
    PE_pb_cell = mcnp.mcnp.Cell(
        cell_id=121,
        material_id=2,
        density=4.78,
        surface_string='-31',
        label='PE_Pb'
    )
    PE_pb_surface = mcnp.mcnp.Surfaces.rpp(surface_id=31, x1=19, x2=29, y1=-7.5, y2=7.5, z1=-11, z2=9)
    PE_pb_material = mcnp.mcnp.Material(
        material_id=2,
        elements=['6000', '1001', '82000'],
        portions=[0.04286, 0.00714, 0.95000],
        labels=['C', 'H', 'Pb'],
        label='PE+Pb'
    )

    # Pb shields
    Pb_cells = [
        mcnp.mcnp.Cell(cell_id=122, material_id=4, density=11.29, surface_string='-32', label='Pb1'),
        mcnp.mcnp.Cell(cell_id=123, material_id=4, density=11.29, surface_string='-33', label='Pb2'),
        mcnp.mcnp.Cell(cell_id=124, material_id=4, density=11.29, surface_string='-34', label='Pb3'),
        mcnp.mcnp.Cell(cell_id=125, material_id=4, density=11.29, surface_string='-35', label='Pb4'),
        mcnp.mcnp.Cell(cell_id=126, material_id=4, density=11.29, surface_string='-36', label='Pb5'),
        mcnp.mcnp.Cell(cell_id=127, material_id=4, density=11.29, surface_string='-37', label='Pb6'),
    ]
    Pb_surfaces = [
        mcnp.mcnp.Surfaces.rpp(surface_id=32, x1=9, x2=19, y1=4, y2=9, z1=-11, z2=9),
        mcnp.mcnp.Surfaces.rpp(surface_id=33, x1=9, x2=19, y1=-9, y2=-4, z1=-11, z2=9),
        mcnp.mcnp.Surfaces.rpp(surface_id=34, x1=19, x2=29, y1=7.5, y2=12.5, z1=-11, z2=9),
        mcnp.mcnp.Surfaces.rpp(surface_id=35, x1=19, x2=29, y1=-12.5, y2=-7.5, z1=-11, z2=9),
        mcnp.mcnp.Surfaces.rpp(surface_id=36, x1=29, x2=34, y1=-15, y2=15, z1=-11, z2=9),
        mcnp.mcnp.Surfaces.rpp(surface_id=37, x1=9, x2=19, y1=-4, y2=4, z1=4, z2=9),
    ]
    Pb_material = mcnp.mcnp.Material(
        material_id=4,
        elements=['82000'],
        portions=[1.0],
        labels=['Pb'],
        label='Pb'
    )

    # Boric Acid shields
    BA_cells = [
        mcnp.mcnp.Cell(cell_id=128, material_id=3, density=1.5, surface_string='-38', label='BA1'),
        mcnp.mcnp.Cell(cell_id=129, material_id=3, density=1.5, surface_string='-39', label='BA2'),
    ]
    BA_surfaces = [
        mcnp.mcnp.Surfaces.rpp(surface_id=38, x1=-26, x2=26, y1=18, y2=28, z1=-11, z2=9),
        mcnp.mcnp.Surfaces.rpp(surface_id=39, x1=-26, x2=26, y1=-28, y2=-18, z1=-11, z2=9),
    ]
    BA_material = mcnp.mcnp.Material(
        material_id=3,
        elements=['1001', '5010', '5011', '8016'],
        portions=[0.048535, 0.034981, 0.139923, 0.776561],
        labels=['H', 'B10', 'B11', 'O'],
        label='Boric Acid'
    )

    # Aluminum shield
    Al_cell = mcnp.mcnp.Cell(
        cell_id=130,
        material_id=6,
        density=2.7,
        surface_string='-40',
        label='Al'
    )
    Al_surface = mcnp.mcnp.Surfaces.rpp(surface_id=40, x1=-65, x2=65, y1=-28, y2=28, z1=10, z2=10.5)
    Al_material = mcnp.mcnp.Material(
        material_id=6,
        elements=['13027'],
        portions=[1.0],
        labels=['Al'],
        label='Al'
    )

    # Fe tube and Fe shields
    Fe_tube_cell = mcnp.mcnp.Cell(
        cell_id=131,
        material_id=7,
        density=7.92,
        surface_string='301 -302 303 -304',
        label='Fe-tube'
    )
    Fe_tube_surfaces = [
        mcnp.mcnp.Surfaces.px(surface_id=301, D=-20),
        mcnp.mcnp.Surfaces.px(surface_id=302, D=18),
        mcnp.mcnp.Surfaces.cx(surface_id=303, r=3.71),
        mcnp.mcnp.Surfaces.cx(surface_id=304, r=3.81),
    ]
    Fe_tube_material = mcnp.mcnp.Material(
        material_id=7,
        elements=['006000.50c', '014000.60c', '015031.50c', '016000.60c', '024000.50c', '025055.60c', '026000.50c', '028000.50c'],
        portions=[0.0004, 0.005, 0.00023, 0.00015, 0.1899981, 0.0099999, 0.7017229, 0.0924991],
        labels=['C', 'Si', 'P', 'S', 'Cr', 'Mn', 'Fe', 'Ni'],
        label='Fe-tube'
    )

    Fe_cells = [
        mcnp.mcnp.Cell(cell_id=132, material_id=5, density=7.8, surface_string='-401', label='Fe'),
        mcnp.mcnp.Cell(cell_id=133, material_id=5, density=7.8, surface_string='-402', label='Fe'),
    ]
    Fe_surfaces = [
        mcnp.mcnp.Surfaces.rpp(surface_id=401, x1=-27, x2=27, y1=29, y2=34, z1=-11, z2=9),
        mcnp.mcnp.Surfaces.rpp(surface_id=402, x1=-27, x2=27, y1=-34, y2=-29, z1=-11, z2=9),
    ]
    Fe_material = mcnp.mcnp.Material(
        material_id=5,
        elements=['26000'],
        portions=[1.0],
        labels=['Fe'],
        label='Fe'
    )

    arch_cells = [PE_pb_cell] + Pb_cells + BA_cells + [Al_cell, Fe_tube_cell] + Fe_cells
    arch_surfaces = [PE_pb_surface] + Pb_surfaces + BA_surfaces + [Al_surface] + Fe_tube_surfaces + Fe_surfaces
    arch_materials = [PE_pb_material, Pb_material, BA_material, Al_material, Fe_tube_material, Fe_material]

    # Wheels
    wheel_cells = [
        mcnp.mcnp.Cell(cell_id=21, material_id=13, density=0.92, surface_string='-41 42', label='Wheel 1'),
        mcnp.mcnp.Cell(cell_id=211, material_id=13, density=0.92, surface_string='-421 422', label='Wheel 1'),
        mcnp.mcnp.Cell(cell_id=212, material_id=13, density=0.92, surface_string='-423 424', label='Wheel 1'),
        mcnp.mcnp.Cell(cell_id=22, material_id=13, density=0.92, surface_string='-43 44', label='Wheel 2'),
        mcnp.mcnp.Cell(cell_id=221, material_id=13, density=0.92, surface_string='-441 442', label='Wheel 2'),
        mcnp.mcnp.Cell(cell_id=222, material_id=13, density=0.92, surface_string='-443 444', label='Wheel 2'),
        mcnp.mcnp.Cell(cell_id=23, material_id=13, density=0.92, surface_string='-45 46', label='Wheel 3'),
        mcnp.mcnp.Cell(cell_id=231, material_id=13, density=0.92, surface_string='-461 462', label='Wheel 3'),
        mcnp.mcnp.Cell(cell_id=232, material_id=13, density=0.92, surface_string='-463 464', label='Wheel 3'),
        mcnp.mcnp.Cell(cell_id=24, material_id=13, density=0.92, surface_string='-47 48', label='Wheel 4'),
        mcnp.mcnp.Cell(cell_id=241, material_id=13, density=0.92, surface_string='-481 482', label='Wheel 4'),
        mcnp.mcnp.Cell(cell_id=242, material_id=13, density=0.92, surface_string='-483 484', label='Wheel 4'),
    ]
    wheel_surfaces = [
        mcnp.mcnp.Surfaces.rcc(surface_id=41, vx=-2, vy=77, vz=8, h1=0.0, h2=25, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=42, vx=-2, vy=77, vz=8, h1=0.0, h2=25, h3=0.0, r=27.7),
        mcnp.mcnp.Surfaces.rcc(surface_id=421, vx=-2, vy=75.7, vz=8, h1=0.0, h2=1.3, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=422, vx=-2, vy=75.7, vz=8, h1=0.0, h2=1.3, h3=0.0, r=15.24),
        mcnp.mcnp.Surfaces.rcc(surface_id=423, vx=-2, vy=102, vz=8, h1=0.0, h2=1.3, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=424, vx=-2, vy=102, vz=8, h1=0.0, h2=1.3, h3=0.0, r=15.24),
        mcnp.mcnp.Surfaces.rcc(surface_id=43, vx=68, vy=77, vz=8, h1=0.0, h2=25, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=44, vx=68, vy=77, vz=8, h1=0.0, h2=25, h3=0.0, r=27.7),
        mcnp.mcnp.Surfaces.rcc(surface_id=441, vx=68, vy=75.7, vz=8, h1=0.0, h2=1.3, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=442, vx=68, vy=75.7, vz=8, h1=0.0, h2=1.3, h3=0.0, r=15.24),
        mcnp.mcnp.Surfaces.rcc(surface_id=443, vx=68, vy=102, vz=8, h1=0.0, h2=1.3, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=444, vx=68, vy=102, vz=8, h1=0.0, h2=1.3, h3=0.0, r=15.24),
        mcnp.mcnp.Surfaces.rcc(surface_id=45, vx=-2, vy=-77, vz=8, h1=0.0, h2=-25, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=46, vx=-2, vy=-77, vz=8, h1=0.0, h2=-25, h3=0.0, r=27.7),
        mcnp.mcnp.Surfaces.rcc(surface_id=461, vx=-2, vy=-75.7, vz=8, h1=0.0, h2=-1.3, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=462, vx=-2, vy=-75.7, vz=8, h1=0.0, h2=-1.3, h3=0.0, r=15.24),
        mcnp.mcnp.Surfaces.rcc(surface_id=463, vx=-2, vy=-102, vz=8, h1=0.0, h2=-1.3, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=464, vx=-2, vy=-102, vz=8, h1=0.0, h2=-1.3, h3=0.0, r=15.24),
        mcnp.mcnp.Surfaces.rcc(surface_id=47, vx=68, vy=-77, vz=8, h1=0.0, h2=-25, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=48, vx=68, vy=-77, vz=8, h1=0.0, h2=-25, h3=0.0, r=27.7),
        mcnp.mcnp.Surfaces.rcc(surface_id=481, vx=68, vy=-75.7, vz=8, h1=0.0, h2=-1.3, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=482, vx=68, vy=-75.7, vz=8, h1=0.0, h2=-1.3, h3=0.0, r=15.24),
        mcnp.mcnp.Surfaces.rcc(surface_id=483, vx=68, vy=-102, vz=8, h1=0.0, h2=-1.3, h3=0.0, r=29),
        mcnp.mcnp.Surfaces.rcc(surface_id=484, vx=68, vy=-102, vz=8, h1=0.0, h2=-1.3, h3=0.0, r=15.24),
    ]
    wheel_materials = [
        mcnp.mcnp.Material(
            material_id=13,
            elements=['1001', '6000'],
            portions=[0.118371, 0.881629],
            labels=['H', 'C'],
            label='Wheel'
        )
    ]

    return (
        (arch_cells, arch_surfaces, arch_materials),
        (det_cells, det_surfaces, det_materials),
        (wheel_cells, wheel_surfaces, wheel_materials)
    )

In [49]:
# join multiple lists into a single list
def join_lists(lists):
    out = []
    for lst in lists:
        out += lst
    return out

In [50]:
arch_chem = [(
    j.label, 
    j.elements, 
    j.labels, 
    j.portions
    ) for j in join_lists([i[-1] for i in Build2025(None, None, None)])]

In [51]:

arch_chem_table = []
for label, elementss, labels, portions in arch_chem:
    for _, (elem, lab, port) in enumerate(zip(elementss, labels, portions)):
        if _ == 0:
            arch_chem_table.append((label, lab, elem, port))
        else:
            arch_chem_table.append(("", lab, elem, port))
arch_chem_df = pd.DataFrame(arch_chem_table, columns=['Material', 'Element', 'ID', 'Weight %'])
# convert weight % to percent and print
arch_chem_df['Weight %'] = arch_chem_df['Weight %'] * 100
print(tabulate.tabulate(arch_chem_df, headers='keys', tablefmt='latex', showindex=False, floatfmt=".4f"))

\begin{tabular}{lllr}
\hline
 Material   & Element   & ID         &   Weight \% \\
\hline
 PE+Pb      & C         & 6000       &     4.2860 \\
            & H         & 1001       &     0.7140 \\
            & Pb        & 82000      &    95.0000 \\
 Pb         & Pb        & 82000      &   100.0000 \\
 Boric Acid & H         & 1001       &     4.8535 \\
            & B10       & 5010       &     3.4981 \\
            & B11       & 5011       &    13.9923 \\
            & O         & 8016       &    77.6561 \\
 Al         & Al        & 13027      &   100.0000 \\
 Fe-tube    & C         & 006000.50c &     0.0400 \\
            & Si        & 014000.60c &     0.5000 \\
            & P         & 015031.50c &     0.0230 \\
            & S         & 016000.60c &     0.0150 \\
            & Cr        & 024000.50c &    18.9998 \\
            & Mn        & 025055.60c &     1.0000 \\
            & Fe        & 026000.50c &    70.1723 \\
            & Ni        & 028000.50c &     9.2499 \\
 Fe      

In [52]:
def air_fill(non_air_cell_ids, wall_ids):
    surf_string = '-200 '+'#'+' #'.join(map(str, non_air_cell_ids))+f" #({wall_ids[0][0]} -{wall_ids[0][-1]} {wall_ids[1][0]} -{wall_ids[1][-1]} {wall_ids[2][0]} -{wall_ids[2][-1]})"
    air_surface = mcnp.mcnp.Surfaces.so(200, r=800)
    air_material = mcnp.mcnp.Material(
        material_id=12,
        elements=['8016', '7014'],
        portions=[0.23, 0.77],
        labels=['O', 'N']
    )
    air_cell = mcnp.mcnp.Cell(
        cell_id = 30,
        material_id = 12,
        density = 1.15e-3,
        surface_string=surf_string
    )
    void_cell = mcnp.mcnp.Cell(
        cell_id=31,
        material_id=0,
        density=None,
        surface_string='200',
        importance_string="imp:n,p 0"
    )
    return (air_cell, air_surface, air_material), (void_cell,)

In [53]:
def build_tallies(det_cells, soil_cells):
    det_cells_ids = [cell.cell_id for cell in det_cells]
    soil_cells_ids = [cell.cell_id for cell in soil_cells]

    t6 = mcnp.mcnp.DetectorTally6(6, detector_cells=det_cells_ids)
    t8 = mcnp.mcnp.DetectorTally8(
        8, 
        detector_cells=det_cells_ids, 
        geb = (-0.026198, 0.059551, -0.037176),
        phl = (1, 6, 1, 0)
        )
    
    t18 = mcnp.mcnp.DetectorTally8(
        18, 
        detector_cells=det_cells_ids, 
        geb = (-0.026198, 0.059551, -0.037176),
        )
    

    t28 = mcnp.mcnp.DetectorTally8(
        28, 
        detector_cells=det_cells_ids, 
        )

    # t6 = mcnp.mcnp.DetectorTally6(6, detector_cells=det_cells_ids, soil_cells=soil_cells_ids)
    # t16 = mcnp.mcnp.SoilTally6(16, soil_cells=soil_cells_ids)

    tallies = [
        t6,
        t8, 
        t18,
        t28,
    ]
    return tallies

In [54]:
def force_n_digits(x, n):
    # if x is less that 10^n, return 0000...x such that the length is n digits, else return x
    if x < 10**n:
        return f'{x:0{n}d}'
    return f'{x}'

def howmany_digits(x):
    # return the number of digits in x
    if x == 0:
        return 1
    return len(str(x))

# Generate Scripts

In [55]:

SIMULATIONS = pd.DataFrame(simulations)

In [56]:
mcnp_identifiers

[6000, 1001, 8016, 14000, 11023, 13027, 19000]

In [57]:
elements

['C', 'H', 'O', 'Si', 'Na', 'Al', 'K']

In [58]:
sim_folder = "../../compute/input/"
_id = 0
n_tests = len(SIMULATIONS)
max_digits = howmany_digits(n_tests)

avg_samples = []
soil_infos = []
for idx, test in tqdm(SIMULATIONS.iterrows(), desc="Generating MCNP scripts", total=n_tests):
    chem_function = test['chem_function']
    density_function = test['density_function']
    res = test['resolution']

    (soil_cells, soil_surfaces, soil_materials), (element_mat, density_mat, sections, section_wall_ids), (walls, wall_ids) = GenSoil( 
        elements = mcnp_identifiers, 
        # elements = elements,
        element_f=chem_function, 
        density_f=density_function, 
        x_min = -75, x_max = 125, x_n = 1, 
        y_min = -100, y_max = 100, y_n = 1, 
        z_min = 42, z_max = 92, z_n = 1, 
        cell_id_min=1000,
        n_samples=100)
    (arch_cells, arch_surfaces, arch_materials), (det_cells, det_surfaces, det_materials), (wheel_cells, wheel_surfaces, wheel_materials) = Build2025(0, 0, 0)

    cell_ids = [cell.cell_id for cell in arch_cells+det_cells+wheel_cells]
    (air_cell, air_surface, air_material), (void_cell,) = air_fill(cell_ids, wall_ids=wall_ids)
    tallies = build_tallies(det_cells, soil_cells)
    mcnpscript = mcnp.mcnp.MCNP(
        cells=[air_cell, void_cell] + arch_cells + det_cells + wheel_cells + soil_cells,
        surfaces=[air_surface]+arch_surfaces+det_surfaces+wheel_surfaces+soil_surfaces,
        materials=[air_material]+arch_materials+det_materials+wheel_materials+soil_materials,
        tallies=tallies,
    )

    serial = str(force_n_digits(_id, max_digits))
    _id += 1
    SIMULATIONS.at[idx, 'serial'] = serial

    label = test['label']
    filename = f"{label}.txt"
    SIMULATIONS.at[idx, 'filename'] = filename


    with open(sim_folder + filename, 'w') as f:
        f.write(mcnpscript.string())

    soil_info = {
        "cell_ids": cell_ids,
        "walls": walls,
        "wall_ids": wall_ids,
        "element_mat": element_mat,
        "density_mat": density_mat,
        "sections": sections,
        "section_wall_ids": section_wall_ids
    }

    soil_infos.append(soil_info)

SOILINFOS = pd.DataFrame(soil_infos)

Generating MCNP scripts: 100%|██████████| 248/248 [00:00<00:00, 1746.28it/s]


# Export

In [59]:
SOILINFOS.index = SIMULATIONS.label

In [60]:
SOILINFOS.to_json('../../soilinfos.json', indent=4)

In [61]:
SOILINFOS

,cell_ids,walls,wall_ids,element_mat,density_mat,sections,section_wall_ids
label,,,,,,,
C,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","([-75.0, 125.0], [-100.0, 100.0], [42.0, 92.0])","([2000000, 2000001], [2000002, 2000003], [2000...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]",[[1.3149999999999997]],"[[-75.0, 125.0, -100.0, 100.0, 42.0, 92.0]]","[[2000000, 2000001, 2000002, 2000003, 2000004,..."
H,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","([-75.0, 125.0], [-100.0, 100.0], [42.0, 92.0])","([2000000, 2000001], [2000002, 2000003], [2000...","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]]",[[1.3149999999999997]],"[[-75.0, 125.0, -100.0, 100.0, 42.0, 92.0]]","[[2000000, 2000001, 2000002, 2000003, 2000004,..."
O,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","([-75.0, 125.0], [-100.0, 100.0], [42.0, 92.0])","([2000000, 2000001], [2000002, 2000003], [2000...","[[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]]",[[1.3149999999999997]],"[[-75.0, 125.0, -100.0, 100.0, 42.0, 92.0]]","[[2000000, 2000001, 2000002, 2000003, 2000004,..."
Si,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","([-75.0, 125.0], [-100.0, 100.0], [42.0, 92.0])","([2000000, 2000001], [2000002, 2000003], [2000...","[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]]",[[1.3149999999999997]],"[[-75.0, 125.0, -100.0, 100.0, 42.0, 92.0]]","[[2000000, 2000001, 2000002, 2000003, 2000004,..."
Na,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","([-75.0, 125.0], [-100.0, 100.0], [42.0, 92.0])","([2000000, 2000001], [2000002, 2000003], [2000...","[[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]]",[[1.3149999999999997]],"[[-75.0, 125.0, -100.0, 100.0, 42.0, 92.0]]","[[2000000, 2000001, 2000002, 2000003, 2000004,..."
...,...,...,...,...,...,...,...
MMica_D1p315_C0p15_H0p0,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","([-75.0, 125.0], [-100.0, 100.0], [42.0, 92.0])","([2000000, 2000001], [2000002, 2000003], [2000...","[[0.15000000000000024, 0.004302264022856995, 0...",[[1.3149999999999997]],"[[-75.0, 125.0, -100.0, 100.0, 42.0, 92.0]]","[[2000000, 2000001, 2000002, 2000003, 2000004,..."
MMica_D1p315_C0p15_H0p05,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","([-75.0, 125.0], [-100.0, 100.0], [42.0, 92.0])","([2000000, 2000001], [2000002, 2000003], [2000...","[[0.15000000000000024, 0.009644526887555465, 0...",[[1.3149999999999997]],"[[-75.0, 125.0, -100.0, 100.0, 42.0, 92.0]]","[[2000000, 2000001, 2000002, 2000003, 2000004,..."
MMica_D1p315_C0p15_H0p1,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","([-75.0, 125.0], [-100.0, 100.0], [42.0, 92.0])","([2000000, 2000001], [2000002, 2000003], [2000...","[[0.15000000000000024, 0.014986789752253966, 0...",[[1.3149999999999997]],"[[-75.0, 125.0, -100.0, 100.0, 42.0, 92.0]]","[[2000000, 2000001, 2000002, 2000003, 2000004,..."


In [62]:
# FUNCTIONS = SIMULATIONS[['label', 'chem_function_name', 'chem_function', 'density_function_name', 'density_function']]
# FUNCTIONS.to_pickle('../../functions.pkl')

In [63]:
SIMS = SIMULATIONS.drop(columns=['chem_function', 'density_function'])

In [64]:
SIMS.to_csv('../../sims.csv')